# Solutions — Forms

Only look here after you've actually tried the exercises in `forms.ipynb`.

### LESSON 30 — Exercise

**Part 1 — a second controlled input.** Four pieces, and all four are needed:

```jsx
const [city, setCity] = useState("");            // 1. the state, starting as a string

function handleCityChange(event) {               // 3. the handler
  setCity(event.target.value);                   // 4. the setter, with the DOM's value
}

<input id="city" value={city} onChange={handleCityChange} />   // 2. value from state
```

`name` and `city` are two independent state variables, so typing in one cannot affect the
other — the same independence two `useState` calls always give you (LESSON 25).

**Part 2 — converting the uncontrolled input.** It already had the handler half; what it was
missing was the instruction half:

```jsx
<input
  id="uncontrolled"
  value={name}
  onChange={(e) => setName(e.target.value)}
/>
```

Now both inputs read from the same `name`, so typing in either updates the state and **both**
redraw from it. Two inputs, one source of truth — which is LESSON 29's idea arriving in a
place you would not have predicted.

Note what had to change in the handler too. Logging `e.target.value` was enough when React
was only listening; controlling it means the value has to be *stored*.

**Part 3 — who owns it.** The controlled inputs change because their displayed text is
`value={name}`, and `setName("Ada")` changed `name` — the button never touched an input, it
changed the state the inputs are drawn from. The uncontrolled input did not change because
nothing in React was ever telling it what to display; its text lives in the DOM, and only
typing into it puts something there.

**Common mistakes.**

- Adding `value={city}` and forgetting `setCity` in the handler. The input freezes — and this
  is the fastest way to feel what "controlled" actually means, because the box genuinely
  stops accepting letters.
- Writing `onChange={handleCityChange()}` with parentheses. That calls the handler during
  render, exactly as in LESSON 22 — and it is worse here, because calling a setter during
  render is how you get an infinite render loop.
- Reusing `name`'s handler for the `city` input. Both boxes then drive the same state and
  the two inputs mirror each other — which is Part 2's goal, and a bug in Part 1.
- Starting the state as `useState()` and adding the string later. See mini challenge 3.

### LESSON 30 — Mini challenge

**1. `<input value="Sam" />`**

*What the user sees:* the box contains `Sam` and refuses every keystroke. The text never
changes, no matter what is typed.

*Which part of the loop is broken:* steps 3 to 5 are missing entirely. There is no handler,
so nothing reads the event and nothing updates any state — and since the displayed value is
the hard-coded string `"Sam"`, every render puts `Sam` straight back. React warns:

```text
You provided a `value` prop to a form field without an `onChange` handler. This will render
a read-only field. If the field should be mutable use `defaultValue`. Otherwise, set either
`onChange` or `readOnly`.
```

*Corrected:*

```jsx
const [name, setName] = useState("Sam");
<input value={name} onChange={(e) => setName(e.target.value)} />
```

**2. `<input value={name} onChange={setName} />`**

*What the user sees:* the moment they type one character, the box fills with the literal text
`[object Object]` — and then behaves oddly for every keystroke after.

*Which part of the loop is broken:* step 4. The handler is called with the **event**, not
with the event's value, so `setName` stores the event object itself. `name` stops being a
string, and `value={name}` renders that object the only way it can — as `[object Object]`.

The dangerous part is that **nothing complains**. No exception, no React warning. The
handler was a function, which is all `onChange` requires, and it was called correctly with
one argument — the argument just was not the one you wanted.

*Corrected:*

```jsx
<input value={name} onChange={(e) => setName(e.target.value)} />
```

This is the whole reason the lesson insists `event.target.value` and the state value are two
different things. `setName` takes the value; `onChange` hands over the event. Somebody has to
do the unwrapping, and that somebody is you.

**3. `useState()` with no initial value**

*What the user sees:* the input looks fine and works fine. The problem shows up in the
console.

*Which part of the loop is broken:* step 1, on the first render only. `useState()` returns
`undefined`, so `value={undefined}` — which React reads as "this input is uncontrolled". The
first keystroke sets a real string, and the input silently changes category:

```text
A component is changing an uncontrolled input to be controlled. This is likely caused by the
value changing from undefined to a defined value, which should not happen. Decide between
using a controlled or uncontrolled input element for the lifetime of the component.
```

*Corrected:*

```jsx
const [name, setName] = useState("");
```

An input is controlled or uncontrolled **for its whole life**, and the initial state is what
decides which. The empty value of a text input is `""`, not "nothing".

**The pattern across all three.** Each one removes a different piece of the same loop:

| | missing | result |
|---|---|---|
| 1 | the handler, and the state | frozen box, React warns |
| 2 | the unwrapping of `.target.value` | `[object Object]`, silent |
| 3 | a string to start from | works, but switches mode and React warns |

If an input misbehaves, walk the loop: is there state, does `value` come from it, does the
handler read `event.target.value`, and does it call the setter?

### LESSON 31 — Exercise

**Part 1.**

In [ ]:
function l31updateField(values, name, value) {
  return { ...values, [name]: value };
}

function l31applyEdits(values, edits) {
  let result = values;
  for (const [name, value] of edits) {
    result = l31updateField(result, name, value);
  }
  return result;
}

const l31blank = { firstName: "", email: "", city: "" };

const l31edits = [
  ["firstName", "Ada"],
  ["city", "Bologna"],
  ["firstName", "Grace"],   // overwrites the first edit
];

const l31final = l31applyEdits(l31blank, l31edits);

console.log("start:", JSON.stringify(l31blank));
console.log("final:", JSON.stringify(l31final));

console.log("start unchanged?  ", JSON.stringify(l31blank) === '{"firstName":"","email":"","city":""}');
console.log("exactly 3 keys?   ", Object.keys(l31final).length === 3, "|", Object.keys(l31final).join(", "));
console.log("last edit won?    ", l31final.firstName === "Grace");

// Each edit builds on the result of the one before, which is why the third edit overwrites
// the first instead of being lost. `values` itself is only ever read.

**3. The mismatch — what the user actually sees.**

In [ ]:
// The input is `name="firstname"`; the state property is `firstName`.
//
// The user types "A". The handler runs normally: event.target.name is "firstname", so the
// new object is { ...values, firstname: "A" } - a brand new property, spelled the way the
// input is spelled. React renders. And then:
//
//   value={values.firstName}   ->  still "", because nothing ever wrote to firstName
//
// So the box appears to reject the keystroke. It looks exactly like a broken or read-only
// input, and the letter never arrives.
//
// Nothing reports a problem because nothing IS a problem, as far as JavaScript is concerned:
// adding a new property to an object literal is legal, and a computed key can be any string.
// There is no schema anywhere saying which properties this object is allowed to have.

const l31mismatch = { ...{ firstName: "", email: "x" }, firstname: "A" };
console.log(JSON.stringify(l31mismatch));
console.log("firstName is still empty:", l31mismatch.firstName === "");

**Part 2 — the playground.** Three changes and no new handler:

```jsx
const [values, setValues] = useState({ firstName: "", email: "", city: "" });

<label>
  city: <input id="city" name="city" value={values.city} onChange={handleChange} />
</label>
```

`handleChange` needed nothing added to it. That is the whole return on this lesson: the
handler does not know or care how many fields there are, because the field names itself.

Misspelling the new input's `name` to `citty` produces exactly the Part 1.3 behaviour — the
logged state grows a `citty` property, and the city box stays empty however much you type.

**Common mistakes.**

- Adding the input but forgetting the field in `useState`. The value is `undefined` on the
  first render, which makes the input uncontrolled — LESSON 30's mistake 3, with React's
  "changing an uncontrolled input to be controlled" warning to match.
- Giving the third input its own handler out of habit. Nothing breaks, but the reason for the
  whole pattern has been thrown away.
- Writing `value={values}` instead of `value={values.city}`. The input receives an object and
  displays `[object Object]` — LESSON 30's mistake 2 wearing a different hat.

### LESSON 31 — Mini challenge

Starting from `{ firstName: "", email: "ada@example.com" }` and typing `A` into `firstName`:

| | resulting state | what the user sees |
|---|---|---|
| 1 `setValues({ [name]: value })` | `{ firstName: "A" }` | the letter appears — and the email field goes blank |
| 2 `values[name] = value; setValues(values)` | `{ firstName: "A", email: "ada@example.com" }` | **nothing happens on screen** |
| 3 `setValues({ ...values, name: value })` | `{ firstName: "", email: "ada@example.com", name: "A" }` | nothing appears in the box |
| 4 `setValues({ ...values, [name]: value })` | `{ firstName: "A", email: "ada@example.com" }` | correct |

**1.** No spread, so there is nothing to preserve — the new object has one property and every
other field is gone. In a real form the user watches a field they already filled in empty
itself.

**2.** The object was edited in place, so the object handed to the setter *is* the object
already in state. `Object.is` matches and React skips the re-render (LESSON 27). The data
did change; the screen did not.

**3.** `name` without brackets is the literal key `"name"`. The state gains a property called
`name`, `firstName` is never written, and the input — reading `values.firstName` — stays
empty.

**Which is hardest to notice:** number 3. Number 1 is loud, because a filled-in field visibly
empties. Number 2 is loud in a different way, because the input freezes completely and you
investigate immediately. Number 3 is quiet: the form still works, no error appears, the
object still looks plausible in a log, and the only symptom is that one field will not accept
input — which is easy to blame on anything else. It is also a one-character difference from
the correct line.

### LESSON 32 — Exercise

**Part 1 — the default, measured.**

```text
URL before:  http://localhost:5392/
URL after:   http://localhost:5392/?firstName=Ada&email=ada%40x.dev
```

The `firstName` and `email` in that URL are the inputs' **`name` attributes** — the same ones
LESSON 31 used to decide which field to update. The browser has always used `name` to label
form data; LESSON 31 borrowed it, and here you see its original job.

What happened to the list: it is empty. The submission was a real navigation — the page
reloaded, the component mounted fresh, and every `useState` went back to its initial value.
The entry was added to state and then thrown away microseconds later, along with everything
else.

That is the failure `preventDefault()` exists to stop, and it is worth having seen once:
"my form submits and the page flashes and everything resets" is a very common first bug, and
this is what it looks like.

**Part 2 — the five checks.**

```jsx
const emptyForm = { firstName: "", email: "", city: "" };

function handleSubmit(event) {
  event.preventDefault();                                    // 3
  setSubmitted([...submitted, { id: crypto.randomUUID(), ...values }]);   // 4
  setValues(emptyForm);                                      // 5
}

<form onSubmit={handleSubmit}>                               {/* 2 */}
  <input name="city" value={values.city} onChange={handleChange} />   {/* 1 */}
  <button type="submit">Add</button>
</form>
```

Only the first two lines of the component changed to add `city`: the field in `emptyForm`,
and the input itself. `handleChange` was untouched again — LESSON 31 still paying for itself.

**Part 3 — why Enter works.** The handler is on the `<form>`, and Enter in a text field asks
the *form* to submit, exactly as the button does. Neither route goes near the button's own
click handling, which is why putting the handler on the form rather than the button is the
version that covers both.

**Common mistakes.**

- Forgetting `type="submit"` on the button — or rather, adding `type="button"` and then
  wondering why nothing happens. A `<button>` inside a form submits by default; `type="button"`
  opts out.
- Calling `preventDefault()` somewhere other than the submit handler, such as in `onChange`.
  Nothing breaks visibly, and nothing is being prevented either.
- Resetting by setting each field individually. It works, but `setValues(emptyForm)` is one
  line and cannot drift out of step when a field is added.
- Writing `const emptyForm = { ... }` *inside* the component. It still works — but a fresh
  object is built every render, and there is no reason for that when the value never changes.

### LESSON 32 — Mini challenge

**1. No `preventDefault`.** The entry is added, the state resets, and then the browser
navigates to `?firstName=…` and reloads the page. The user sees a flash and an empty form with
an empty list — and a URL full of their data. The code above the missing line was all correct,
which is what makes it confusing.

**2. The handler on the button.** `handleSubmit` runs — a click on a submit button does fire
its `onClick` — so the entry is added and the state reset. Then the form submits anyway,
because nothing called `preventDefault()`, and the page reloads exactly as in 1. Moving the
handler to the button does not remove the form's default behaviour; it just puts your code
somewhere that cannot easily stop it, since the `event` here is the click, not the submit.

**3. `stopPropagation()` instead of `preventDefault()`.** Also a reload. These two are
unrelated, as LESSON 23 said: `stopPropagation` controls which *handlers* see the event, and
there are no other handlers here to stop. The browser's default action is untouched.

**4. Reset before add — and it works.** The list gets `Ada — ada@x.dev`, correctly.

```js
event.preventDefault();
setValues(emptyForm);                  // schedules a change
addEntry({ ...values });               // values is STILL this render's values
```

`setValues` does not reach back and change the `values` variable the handler is holding —
LESSON 26, in a place you would not have gone looking for it. Both lines read the same fixed
object, so the order genuinely does not matter here.

It is still worth writing add-then-reset, because it says what you mean. The day someone
changes `addEntry({ ...values })` to read from somewhere else, the order will start mattering
and the reasoning will not be on the screen any more.

**What all four have in common.** Three of them produce the identical symptom — a page reload
that destroys everything — from three different causes: a missing call, a handler in the wrong
place, and the wrong method. When a React form flashes and empties itself, the question is
never "which line is wrong" but "did anything actually prevent the browser's default?"

### LESSON 33 — Exercise

**Part 1.** The same contract, one more field.

In [ ]:
function l33validateWithAge(values) {
  const errors = {};

  const firstName = values.firstName.trim();
  if (firstName === "") {
    errors.firstName = "First name is required.";
  } else if (firstName.length < 2) {
    errors.firstName = "First name must be at least 2 characters.";
  }

  const email = values.email.trim();
  if (email === "") {
    errors.email = "Email is required.";
  } else if (!email.includes("@") || email.startsWith("@") || email.endsWith("@")) {
    errors.email = "Email must look like name@example.com.";
  }

  const age = values.age.trim();
  if (age === "") {
    errors.age = "Age is required.";
  } else if (!Number.isFinite(Number(age))) {
    errors.age = "Age must be a number.";
  } else if (Number(age) < 18 || Number(age) > 120) {
    errors.age = "Age must be between 18 and 120.";
  }

  return errors;
}

const l33ok = { firstName: "Ada", email: "ada@example.com", age: "36" };
const l33bad = { firstName: "", email: "nope", age: "abc" };

console.log("valid      ->", JSON.stringify(l33validateWithAge(l33ok)));
console.log("all wrong  ->", JSON.stringify(l33validateWithAge(l33bad)));

console.log("1 valid gives {}?    ", Object.keys(l33validateWithAge(l33ok)).length === 0);
console.log("2 exactly 3 keys?    ", Object.keys(l33validateWithAge(l33bad)).length === 3);
console.log("3 deterministic?     ",
  JSON.stringify(l33validateWithAge(l33bad)) === JSON.stringify(l33validateWithAge(l33bad)));

const l33snapshot = JSON.stringify(l33bad);
l33validateWithAge(l33bad);
console.log("4 values unchanged?  ", JSON.stringify(l33bad) === l33snapshot);

// A few edge cases worth checking, because they are the ones that bite:
console.log("empty age is 'required', not 'must be a number':",
  l33validateWithAge({ ...l33ok, age: "" }).age);
console.log("'17' is out of range:", l33validateWithAge({ ...l33ok, age: "17" }).age);
console.log("' 36 ' is fine:      ", JSON.stringify(l33validateWithAge({ ...l33ok, age: " 36 " })));

The order of the `age` checks matters: empty first, then "is it a number", then the range.
Reversed, an empty string would be reported as out of range — `Number("")` is `0` — which is
a confusing message for a field the user simply has not filled in yet.

**3. Why messages rather than showing them.**

In [ ]:
// Because deciding and displaying are different jobs, and only one of them is testable in a
// single line.
//
// A validator that returns messages can be called from anywhere: a submit handler, a test, a
// second form, or a notebook cell like this one. It has no opinion about React, about which
// element the message goes next to, about styling, or about whether the message is shown at
// all.
//
// The moment it shows the errors itself it needs state, which means it needs to be inside a
// component, which means it can only be exercised by rendering that component and clicking
// its button. The same logic - and it is the logic that is worth getting right - becomes
// several times harder to check.

console.log("validate decides; the component displays");

**Part 2 — the playground.** The additions are exactly the two from LESSON 30–31, plus the
message:

```jsx
const emptyForm = { firstName: "", email: "", age: "" };

<p>
  <label>
    age: <input id="age" name="age" value={values.age} onChange={handleChange} />
  </label>
  {errors.age && <span style={{ color: "crimson" }}> {errors.age}</span>}
</p>
```

`handleChange` was untouched (LESSON 31) and `handleSubmit` was untouched (LESSON 32) — it
calls `validate`, stores the result, and stops if there is anything in it. Adding a field
changed the form and the validator, and nothing in between. That is what keeping the two jobs
apart buys you.

Submitting with `abc` gives "Age must be a number."; with `9`, "Age must be between 18 and
120."; with `30`, the entry is added and the form resets.

**Common mistakes.**

- Returning `null` or `false` for "valid" instead of `{}`. Every caller then needs to check
  which of three shapes it got, and `Object.keys(errors).length === 0` stops working.
- Calling `validate` on every keystroke and showing the result. The user is told their email
  is invalid after typing one character. Submit-time is the right default.
- Forgetting that input values are always **strings**. `values.age > 18` compares a string to
  a number and works by accident often enough to be dangerous; `Number(age)` makes it explicit.
- Putting the messages in the validator but the *rules* in the component (`if (!errors.email
  && somethingElse)`). The rules then live in two places, which is the thing this lesson is
  trying to prevent.

### LESSON 33 — Mini challenge

**1. It changes the values it was given.**

```js
values.firstName = values.firstName.trim();
```

This breaks "it minds its own business", and because `values` is the object in React state, it
is also LESSON 27's mutation bug: state has been edited in place, behind React's back. The
user sees their typing silently altered — trailing spaces vanishing as they submit — and
anything comparing previous values with next values is now comparing one object with itself.
Fix: `const firstName = values.firstName.trim();` and leave `values` alone.

**2. It returns a boolean.**

The contract is broken, not the purity. It is a perfectly pure function — but it can only say
*something is wrong*, never *what* or *where*. The component cannot put a message next to the
right input, so the best it can do is one generic line for the whole form. Fix: return the
errors object, and derive the boolean from it when you need one.

**3. It is not deterministic.**

`new Date()` means the same input produces different output depending on when you call it —
so "same inputs, same output" fails. Two consequences: the function cannot be checked without
controlling the clock, and the form's behaviour changes under a user at 17:00 with no input
from them at all.

**4. It does the displaying too.**

Taking `setErrors` as an argument welds the validator to one component's state, and it returns
nothing, so nobody else can use its result. It is no longer a validator; it is part of a
submit handler that happens to live in another file. Fix: return the errors and let the caller
decide what to do with them.

**Where the 5pm rule belongs.**

It is a real rule, and it is not validation *of the values* — it does not depend on what the
user typed at all. It belongs in the submit handler, alongside the validation call:

```js
function handleSubmit(event) {
  event.preventDefault();

  const found = validate(values);
  setErrors(found);
  if (Object.keys(found).length > 0) return;

  if (new Date().getHours() > 17) {
    setFormMessage("Submissions close at 5pm.");
    return;
  }

  addEntry({ ...values });
}
```

The distinction is worth keeping: `validate` answers "are these values acceptable?", which
only ever depends on the values. "May this be submitted right now?" depends on the world, and
things that depend on the world belong where the action happens. Keeping the clock out of
`validate` is what lets you check it with `l33validate(someValues)` and trust the answer.